# Kahnn `nano` — Colab GPU (T4/L4)

Train **nano** (~1.8M, ~37M tok) with `train_universal.py` on free Colab GPU.

1. **Runtime → Change runtime type → GPU** (T4/L4). If Colab gives CPU only, cell 0 stops with a clear FR/EN message.
2. Drive is **optional** (`USE_DRIVE=False` by default) — ckpts live under `/content` until you sync.
3. Repo is **public** (`AFKmoney/kahnn`); clone needs no token.
4. Corpus: Drive copy **or** minimal Gutenberg rebuild (no interactive upload required).

CPU box ref (2026-09-08): ~**1.07–1.23k tok/s**. T4 should be **much faster** — **measure `tps=` on smoke**; no invented GPU bench. See `docs/COLAB_GPU.md`.


## 0 — GPU check (fail clear if no GPU)


In [ ]:
import torch, subprocess, sys
print(torch.__version__, "cuda_available=", torch.cuda.is_available())
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found")
if not torch.cuda.is_available():
    msg_en = (
        "NO GPU: Colab gave a CPU runtime. Runtime → Change runtime type → GPU. "
        "Free tier often has no GPU quota left — try later, switch account, or Colab Pro."
    )
    msg_fr = (
        "PAS de GPU: Colab a donné un runtime CPU. Runtime → Modifier le type d'exécution → GPU. "
        "Le gratuit n'a souvent plus de quota GPU — réessaie plus tard, autre compte, ou Colab Pro."
    )
    print("\n=== EN ===\n" + msg_en)
    print("\n=== FR ===\n" + msg_fr)
    raise SystemExit("Stop: no CUDA GPU — fix runtime before training.")
print("GPU:", torch.cuda.get_device_name(0))


## 1 — Deps (keep Colab torch CUDA; do not pip install CPU torch)


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tiktoken", "tqdm", "numpy"])
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])
except Exception as e:
    print("bitsandbytes optional — skip OK:", e)
import tiktoken, tqdm, numpy, torch
print("ok", torch.__version__, torch.cuda.is_available())


## 2 — Clone repo (public) + optional Drive


In [ ]:
from pathlib import Path
import subprocess, shutil, sys, os

REPO = Path("/content/kahnn")
DRIVE = Path("/content/drive/MyDrive/kahnn")
BRANCH = "main"  # pin main after merge; override only if needed
USE_DRIVE = False  # set True to mount Drive for ckpt persistence
REPO_URL = "https://github.com/AFKmoney/kahnn.git"

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        (DRIVE / "runs/nano_colab").mkdir(parents=True, exist_ok=True)
        (DRIVE / "data").mkdir(parents=True, exist_ok=True)
        print("Drive mounted:", DRIVE)
    except Exception as e:
        print("Drive mount failed (continuing without Drive):", e)
        USE_DRIVE = False
else:
    print("USE_DRIVE=False — ckpts stay under /content (ephemeral). Set USE_DRIVE=True + re-run to persist.")

if REPO.exists() and not (REPO / ".git").exists():
    shutil.rmtree(REPO)

if not (REPO / ".git").exists():
    cmd = ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)]
    print("Running:", " ".join(cmd))
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as e:
        print("Clone failed.")
        print("EN: Repo is public at", REPO_URL, "— check network / branch name", BRANCH)
        print("FR: Le repo est public — vérifie le réseau / le nom de branche", BRANCH)
        print("If you forked a private copy, use a token URL or upload a zip instead.")
        raise SystemExit(e.returncode)
else:
    print("Repo already present:", REPO)

os.chdir(REPO)
print("HEAD", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 3 — Corpus (Drive copy → rebuild; no blocking files.upload)


In [ ]:
from pathlib import Path
import shutil
DATA = Path("/content/kahnn/data"); DATA.mkdir(parents=True, exist_ok=True)
CORPUS = DATA / "corpus.txt"
src = Path("/content/drive/MyDrive/kahnn/data/corpus.txt")
if not CORPUS.exists() and src.exists():
    shutil.copy2(src, CORPUS); print("copied from Drive")
# Optional interactive upload — OFF by default (blocks the cell in headless runs)
DO_UPLOAD = False
if not CORPUS.exists() and DO_UPLOAD:
    try:
        from google.colab import files
        up = files.upload()
        for n, b in up.items():
            CORPUS.write_bytes(b); break
    except Exception as e:
        print("upload skipped", e)
print("corpus", CORPUS.exists(), CORPUS.stat().st_size if CORPUS.exists() else None)
if not CORPUS.exists():
    print("No corpus yet — run the next rebuild cell (Gutenberg, no Drive needed).")


### Rebuild minimal if needed (no Drive)


In [ ]:
from pathlib import Path
import urllib.request, time
CORPUS = Path("/content/kahnn/data/corpus.txt")
RAW = Path("/content/kahnn/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
IDS = [("1342","pride.txt"),("11","alice.txt"),("84","frank.txt"),("1661","sherlock.txt"),("2701","moby.txt")]

def fetch(gid, name):
    out = RAW / name
    if out.exists() and out.stat().st_size > 10000:
        return out
    err = None
    for url in [
        f"https://www.gutenberg.org/files/{gid}/{gid}-0.txt",
        f"https://www.gutenberg.org/ebooks/{gid}.txt.utf-8",
    ]:
        try:
            print("GET", url)
            urllib.request.urlretrieve(url, out)
            if out.stat().st_size > 10000:
                return out
        except Exception as e:
            err = e; time.sleep(0.5)
    raise RuntimeError(err)

if CORPUS.exists() and CORPUS.stat().st_size > 1_000_000:
    print("skip rebuild", CORPUS.stat().st_size)
else:
    parts = [fetch(g, n).read_text(encoding="utf-8", errors="ignore") for g, n in IDS]
    code_parts = []
    root = Path("/content/kahnn")
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in {".py", ".sh", ".md"} and not any(
            x in p.parts for x in ("data", "runs", ".git", "__pycache__", "notebooks")
        ):
            code_parts.append(p.read_text(encoding="utf-8", errors="ignore"))
    base = "\n\n".join(parts + ["\n".join(code_parts)])
    CORPUS.write_text(base * 3, encoding="utf-8")
    print("wrote", CORPUS.stat().st_size)
    d = Path("/content/drive/MyDrive/kahnn/data/corpus.txt")
    if d.parent.exists():
        try:
            d.write_bytes(CORPUS.read_bytes()); print("also saved to Drive")
        except Exception as e:
            print("Drive save skipped", e)


## 4 — Optional resume from Drive / CPU box (`ckpt_1500+`)


In [ ]:
from pathlib import Path
import shutil
OUT = Path("/content/kahnn/runs/nano_colab"); OUT.mkdir(parents=True, exist_ok=True)
RESUME = None  # e.g. "/content/drive/MyDrive/kahnn/runs/from_cpu/ckpt_1500.pt"
drv = Path("/content/drive/MyDrive/kahnn/runs/nano_colab")
if drv.exists():
    for p in drv.glob("ckpt_*.pt"):
        if not (OUT / p.name).exists():
            shutil.copy2(p, OUT / p.name)
if RESUME:
    rp = Path(RESUME); local = OUT / rp.name
    if rp.exists() and rp.resolve() != local.resolve():
        shutil.copy2(rp, local); RESUME = str(local)
    print("resume", RESUME, Path(RESUME).stat().st_size if Path(RESUME).exists() else None)
else:
    print("cold start")
print(sorted(p.name for p in OUT.glob("ckpt_*.pt")))


## 5 — Smoke (first loss/tps) — measure here vs ~1.2k CPU


In [ ]:
import os, subprocess
os.chdir("/content/kahnn")
subprocess.run(["nvidia-smi", "-L"], check=False)
cmd = [
    "python", "train_universal.py",
    "--data", "/content/kahnn/data/corpus.txt",
    "--output", "/content/kahnn/runs/nano_colab",
    "--config", "nano", "--device", "cuda",
    "--micro-batch", "8", "--seq-len", "256", "--grad-accum", "2",
    "--log-every", "1", "--checkpoint-every", "500", "--smoke-steps", "20",
]
print(" ".join(cmd))
subprocess.check_call(cmd)


## 6 — Full nano train (or resume). OOM → micro-batch 4 + `--mod` + `--activation-checkpointing`


In [ ]:
import os, subprocess
from pathlib import Path
os.chdir("/content/kahnn")
OUT = "/content/kahnn/runs/nano_colab"
resume = ""
try:
    RESUME
except NameError:
    RESUME = None
if RESUME:
    resume = f"--resume {RESUME}"
else:
    numbered = []
    for p in Path(OUT).glob("ckpt_*.pt"):
        if p.stem.startswith("ckpt_") and p.stem[5:].isdigit():
            numbered.append((int(p.stem[5:]), p))
    if numbered:
        resume = f"--resume {sorted(numbered)[-1][1]}"
        print(resume)
cmd = (
    f"python train_universal.py --data /content/kahnn/data/corpus.txt "
    f"--output {OUT} --config nano --device cuda --micro-batch 8 "
    f"--seq-len 256 --grad-accum 2 --log-every 10 --checkpoint-every 500 {resume}"
)
print(cmd)
subprocess.check_call(cmd, shell=True)


## 7 — Save ckpts to Drive (skip if Drive missing)


In [ ]:
from pathlib import Path
import shutil
src = Path("/content/kahnn/runs/nano_colab")
dst = Path("/content/drive/MyDrive/kahnn/runs/nano_colab")
if not Path("/content/drive/MyDrive").exists():
    print("Drive not mounted — skip sync. Re-run cell 2 with USE_DRIVE=True first.")
else:
    dst.mkdir(parents=True, exist_ok=True)
    for p in sorted(src.glob("ckpt_*.pt")):
        shutil.copy2(p, dst / p.name); print("copied", p.name, p.stat().st_size)
    if (src / "train_universal.log").exists():
        shutil.copy2(src / "train_universal.log", dst / "train_universal.log")
    for p in sorted(dst.glob("*")):
        print(p.name, p.stat().st_size)


## 8 — teach.py after pretrain


In [ ]:
import os, subprocess
from pathlib import Path
os.chdir("/content/kahnn")
CKPT = "/content/kahnn/runs/nano_colab/ckpt_final.pt"
if not Path(CKPT).exists():
    numbered = [
        (int(p.stem[5:]), p)
        for p in Path("/content/kahnn/runs/nano_colab").glob("ckpt_*.pt")
        if p.stem.startswith("ckpt_") and p.stem[5:].isdigit()
    ]
    if numbered:
        CKPT = str(sorted(numbered)[-1][1])
print("Using", CKPT)
subprocess.check_call([
    "python", "teach.py", "teach",
    "--text", "La capitale du Canada est Ottawa.",
    "--resume", CKPT, "--config", "nano", "--device", "cuda",
    "--output", "/content/kahnn/runs/teach_colab",
])
subprocess.check_call([
    "python", "teach.py", "probe",
    "--text", "La capitale du Canada est",
    "--resume", "/content/kahnn/runs/teach_colab/ckpt_teach.pt",
    "--config", "nano", "--device", "cuda",
])


## Troubleshooting

| Symptom | Fix |
|---------|-----|
| Cell 0 exits / `cuda_available=False` | Runtime → GPU; free quota exhausted → wait / Pro |
| Drive mount hangs / wants interactive auth | Keep `USE_DRIVE=False`; train on `/content`; download ckpts manually |
| `git clone` 404 / auth | Repo is public — wrong branch? pin `BRANCH=\"main\"`. Private fork needs token URL |
| `files.upload()` blocks forever | Leave `DO_UPLOAD=False`; use rebuild cell |
| OOM on T4 | `--micro-batch 4`, `--mod`, `--activation-checkpointing` |
| tiktoken `ValueError` on special EOT token in corpus | Fixed via `data.encode_text` / `encode_ordinary` — pull latest `main` |
| Session died mid-train | Remount Drive (if used) or re-upload last `ckpt_*.pt`, set `RESUME=...` |

Never invent GPU tps; read smoke logs. Details: `docs/COLAB_GPU.md`.
